In [ ]:
# merge all dc8 nasa lfight tracks into one netcdf 
#!pip install Counter 

import glob, pandas as pd, monetio as mio, os
from collections import Counter
import xarray as xr



In [ ]:
DC8_DIR = "/glade/campaign/acom/acom-weather/emmons/ASIAAQ_obs/DC8"

files = sorted(glob.glob(DC8_DIR + "/*.ict")) or sorted(glob.glob(DC8_DIR + "/*"))

def header_cols(path):
    with open(path) as fh:
        lines = fh.read().splitlines()
    nlhead = int(lines[0].split(",")[0])
    return [c.strip() for c in lines[nlhead - 1].split(",")]

cnt = {k: Counter() for k in ["o3", "no2", "no", "alt", "gps", "h2o", "ch4", "co2"]}
for f in files:
    for c in header_cols(f):
        cl = c.lower()
        if "o3" in cl or "ozone" in cl:            cnt["o3"][c]  += 1
        if "no2" in cl:                            cnt["no2"][c] += 1
        elif cl.startswith("no_") or cl == "no":   cnt["no"][c]  += 1
        if "alt" in cl:                            cnt["alt"][c] += 1
        if "h2o" in cl or "water" in cl:           cnt["h2o"][c] += 1
        if "ch4" in cl:                            cnt["ch4"][c] += 1
        if "co2" in cl:                            cnt["co2"][c] += 1
for k, c in cnt.items():
    print(f"{k:5s}:", dict(c))   # shows each candidate name and how many of the N files use it


In [ ]:
DC8_DIR = "/glade/campaign/acom/acom-weather/emmons/ASIAAQ_obs/DC8"

OUT = "/glade/u/home/lcthompson/mm/MELODIES-MONET/docs/examples/ungridded_support/unstructured_grid_read_uxarray/asiaaq_cs_06082026/preprocessing/dc8_data/asiaaq_dc8_merge_all.nc"

ALIASES = {
    "latitude":     ["Latitude_BENNETT"],
    "longitude":    ["Longitude_BENNETT"],
    "pressure_obs": ["Static_Pressure_BENNETT"],
    "altitude":     ["GPS_Altitude_m_DIGANGI", "GPS_Altitude_m", "G_ALT_GATEBE"],
    "temperature":  ["T_GATEBE"],
    "u": ["U_GATEBE"], "v": ["V_GATEBE"], "w": ["W_GATEBE"],
    "h2o": ["H2O_DLH_DISKIN"],
    "O3":  ["O3_ppbv_FRANCHIN",  "O3_ppbv"],     # ppbv
    "NO":  ["NO_pptv_FRANCHIN",  "NO_pptv"],     # pptv
    "NO2": ["NO2_pptv_FRANCHIN", "NO2_pptv"],    # pptv
    "CO":  ["CO_DACOM_DISKIN"],                  # ppbv
    "CH4": ["CH4_DACOM_DISKIN"],                 # ppbv
    "CO2": ["CO2_7000_ppm_DISKIN"],              # ppm
}

NA = [-999999, -99999, -9999, -8888, -7777]

def read_merge(path):
    with open(path) as fh:
        lines = fh.read().splitlines()
    nlhead = int(lines[0].split(",")[0])
    y, m, d = (int(x) for x in lines[6].split(",")[:3])
    base = pd.Timestamp(year=y, month=m, day=d)

    raw = [c.strip() for c in lines[nlhead - 1].split(",")]
    seen, cols = {}, []
    for n in raw:
        if n in seen:
            seen[n] += 1; cols.append(f"{n}.{seen[n]}")
        else:
            seen[n] = 0;  cols.append(n)

    # canonical first alias present in file
    # all aliases present in the file for each canonical name
    present = {
        canon: [c for c in cands if c in cols]
        for canon, cands in ALIASES.items()
    }
    
    usecols = ["Time_Start"] + [
        c for cs in present.values() for c in cs
    ]
    
    df = pd.read_csv(
        path,
        skiprows=nlhead,
        names=cols,
        usecols=usecols,
        na_values=NA,
    )
    
    # choose the first alias that actually contains data
    out = pd.DataFrame({"Time_Start": df["Time_Start"]})
    
    for canon, cs in present.items():
        for c in cs:
            if df[c].notna().any():
                out[canon] = df[c]
                break
    
    df = out
    
    df["time"] = base + pd.to_timedelta(
        pd.to_numeric(df["Time_Start"], errors="coerce"), unit="s")
    return df.drop(columns=["Time_Start"])
    

In [ ]:
keep = list(ALIASES)
files = sorted(glob.glob(os.path.join(DC8_DIR, "asiaaq-mrg10_dc8_*.ict")))
df = pd.concat([read_merge(f) for f in files], ignore_index=True).sort_values("time")

df["pressure_obs"] = df["pressure_obs"] * 100.0       # hPa -> Pa
df = df.dropna(subset=["time"]).drop_duplicates(subset="time").sort_values("time")
ds = df.set_index("time").to_xarray()
ds.to_netcdf(OUT)
print(ds)


In [ ]:
files = sorted(glob.glob(DC8_DIR + "/*"))
f0331 = [x for x in files if "20240331" in x]
print("file:", f0331)
cols = header_cols(f0331[0])     # the header_cols() helper from before
for k in ["o3", "ozone", "no2", "alt", "gps"]:
    print(f"{k:6s}:", [c for c in cols if k in c.lower()])

In [ ]:

ds = xr.open_dataset("/glade/u/home/lcthompson/mm/MELODIES-MONET/docs/examples/ungridded_support/unstructured_grid_read_uxarray/asiaaq_cs_06082026/output/asiaaq_dc8_cam-chem-se-era5.nc4")

df = ds.to_dataframe()
for v in ["altitude","O3","O3_new","NO2","NO2_new","CO","CO_new",
          "temperature","T","pressure_obs"]:
    if v in df:
        print(f"{v:14s} nonnull {int(df[v].notna().sum()):6d} / {len(df)}")

In [ ]:
ds = xr.open_dataset("/glade/u/home/lcthompson/mm/MELODIES-MONET/docs/examples/ungridded_support/unstructured_grid_read_uxarray/asiaaq_cs_06082026/preprocessing/dc8_data/asiaaq_dc8_merge_all.nc")

df = ds.to_dataframe().reset_index()
df["time"] = pd.to_datetime(df["time"])
win = df[(df["time"] >= "2024-03-29") & (df["time"] <= "2024-03-31 23:59")]

print("=== whole file ===")
for v in ["O3","NO2","altitude","CO","temperature", "pressure_obs"]:
    if v in df:
        print(f"{v:12s} {int(df[v].notna().sum()):6d} / {len(df)}")

print("=== 03-29..03-31 window ===")
for v in ["O3","NO2","altitude","CO","temperature", "pressure_obs"]:
    if v in win:
        print(f"{v:12s} {int(win[v].notna().sum()):6d} / {len(win)}")

In [ ]:
# code above checks for the first alias. debug to check for first alias with data... 

f = "/glade/campaign/acom/acom-weather/emmons/ASIAAQ_obs/DC8/asiaaq-mrg10_dc8_20240331_RA_20260509.ict"
NA = [-999999, -99999, -9999, -8888, -7777]

with open(f) as fh:
    lines = fh.read().splitlines()
nlhead = int(lines[0].split(",")[0])
raw = [c.strip() for c in lines[nlhead - 1].split(",")]
seen, cols = {}, []                      # de-dup exactly like read_merge
for n in raw:
    if n in seen:
        seen[n] += 1; cols.append(f"{n}.{seen[n]}")
    else:
        seen[n] = 0;  cols.append(n)

raw_df = pd.read_csv(f, skiprows=nlhead, names=cols, na_values=NA)

print("--- altitude candidates ---")
for c in ["GPS_Altitude_m_DIGANGI","GPS_Altitude_m","G_ALT_GATEBE","Pressure_Altitude_BENNETT"]:
    if c in raw_df: print(f"{c:28s} {int(raw_df[c].notna().sum()):6d} / {len(raw_df)}")

print("--- O3 / NO2 candidates ---")
for c in ["O3_ppbv_FRANCHIN","O3_ppbv","O3_ROZE","NO2_pptv_FRANCHIN","NO2_pptv","NO2_CAESAR","NO2_CANOE"]:
    if c in raw_df: print(f"{c:28s} {int(raw_df[c].notna().sum()):6d} / {len(raw_df)}")